# 09 Inverse RL: Learning Reward Functions from Expert Trades

## 论文参考
- **作者**: Weipeng Zhang
- **年份**: 2023
- **标题**: *Reinforcement Learning for Stock Prediction and HFT with T+1 Rules*
- **摘要**: 本文提出基于逆强化学习(IRL)的交易策略框架。给定一个已知盈利的"专家策略"
  的交易轨迹，通过Maximum Entropy IRL推断出隐含的奖励函数，然后用该奖励函数
  训练新的RL智能体。这种方法特别适用于难以手工设计奖励函数的场景。

> **⚠️ 教学演示声明**
> 
> 本 Notebook 为量化策略算法教学样例，回测结果包含**前视偏差 (Look-ahead Bias)**：
> - 训练标签使用了未来收益率（`shift(-N)`）
> - StandardScaler 等预处理可能在全量数据上拟合
> - 模型参数选择可能基于完整样本期
> 
> **所有回测收益率仅供学习参考，不代表策略的实际可期望表现，不可直接用于实盘交易。**

## 算法原理

### 1. 逆强化学习 (Inverse RL)
- **正向RL**: 给定奖励函数 $R(s,a)$ → 学习最优策略 $\pi^*$
- **逆向RL**: 给定专家策略 $\pi_E$ (演示轨迹) → 推断奖励函数 $R(s,a)$

### 2. Maximum Entropy IRL (MaxEnt IRL)
假设奖励函数是特征的线性组合:
$$R_\theta(s) = \theta^T \phi(s)$$

专家策略服从最大熵原则:
$$P(\tau | \theta) \propto \exp\left(\sum_t R_\theta(s_t)\right)$$

优化目标 (最大化专家轨迹的对数似然):
$$\max_\theta \sum_{\tau \in \mathcal{D}_E} \log P(\tau | \theta)$$

梯度更新:
$$\nabla_\theta \mathcal{L} = \bar{\phi}_E - \mathbb{E}_{\pi_\theta}[\bar{\phi}]$$
即：专家特征期望 - 当前策略下的特征期望

### 3. 流程
1. **专家轨迹**: 从一个盈利策略（如MA金叉）收集交易记录
2. **特征提取**: 计算状态特征 $\phi(s)$
3. **MaxEnt IRL**: 迭代更新 $\theta$ 直到特征匹配
4. **正向训练**: 用学到的 $R_\theta$ 训练新RL智能体
5. **泛化**: 新智能体可能在样本外也表现良好

In [ ]:
# === Cell 3: 专家轨迹 -> 奖励景观 -> 模仿智能体 动画 ===
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

np.random.seed(42)

# Simulate expert trajectory in 2D feature space
n_expert_steps = 50
expert_feat1 = np.cumsum(np.random.randn(n_expert_steps) * 0.3)  # momentum
expert_feat2 = np.cumsum(np.random.randn(n_expert_steps) * 0.2)  # vol signal

# Simulate IRL learning the reward landscape
n_irl_iters = 20
theta = np.array([0.0, 0.0])  # initial reward weights

# True expert feature expectations
expert_feat_exp = np.array([expert_feat1.mean(), expert_feat2.mean()])

frames = []
grid_x = np.linspace(-3, 3, 30)
grid_y = np.linspace(-3, 3, 30)
X_grid, Y_grid = np.meshgrid(grid_x, grid_y)

for iteration in range(n_irl_iters):
    # Current reward landscape
    R = theta[0] * X_grid + theta[1] * Y_grid

    # Simulated learner policy feature expectations (random walk toward expert)
    noise = np.random.randn(2) * 0.5 * np.exp(-0.15 * iteration)
    learner_feat_exp = expert_feat_exp * (1 - np.exp(-0.2 * iteration)) + noise

    # MaxEnt IRL gradient: expert - learner
    grad = expert_feat_exp - learner_feat_exp
    theta += 0.3 * grad

    # Learner trajectory (imitating agent)
    n_imitate = min(iteration * 3 + 5, n_expert_steps)
    imit_feat1 = expert_feat1[:n_imitate] + np.random.randn(n_imitate) * 0.5 * np.exp(-0.1 * iteration)
    imit_feat2 = expert_feat2[:n_imitate] + np.random.randn(n_imitate) * 0.5 * np.exp(-0.1 * iteration)

    frames.append(go.Frame(
        data=[
            go.Contour(x=grid_x, y=grid_y, z=R,
                       colorscale='RdYlGn', name='奖励景观',
                       showscale=True, colorbar=dict(title='R(s)')),
            go.Scatter(x=expert_feat1, y=expert_feat2, mode='lines+markers',
                       name='专家轨迹', line=dict(color='blue', width=2),
                       marker=dict(size=5)),
            go.Scatter(x=imit_feat1, y=imit_feat2, mode='lines+markers',
                       name='模仿智能体', line=dict(color='red', width=2, dash='dot'),
                       marker=dict(size=5, symbol='diamond')),
        ],
        name=f'IRL Iter {iteration+1}'
    ))

fig = go.Figure(data=frames[0].data, frames=frames)
fig.update_layout(
    title=dict(text='Inverse RL: 专家轨迹 -> 奖励景观学习 -> 模仿智能体'),
    xaxis_title='特征1 (动量)', yaxis_title='特征2 (波动)',
    height=600, width=800,
    updatemenus=[dict(type='buttons', buttons=[
        dict(label='▶ 播放', method='animate',
             args=[None, {'frame': {'duration': 500}}]),
        dict(label='⏸ 暂停', method='animate',
             args=[[None], {'frame': {'duration': 0}, 'mode': 'immediate'}]),
    ])],
    sliders=[dict(
        steps=[dict(args=[[f.name]], label=f.name, method='animate') for f in frames],
    )],
)
fig.show()

In [ ]:
# === Cell 4: 导入与配置 ===
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import deque
import random

from shared.data_fetcher import get_stock_daily, get_index_daily
from shared.factors import rsi, macd, sma, momentum, volatility
from shared.backtest_engine import Backtester
from shared.visualization import (plot_equity_curve, plot_drawdown,
                                   plot_metrics_table, plot_cumulative_comparison)
from shared.animation_helpers import animate_rl_trading
from shared.constants import *

print('All imports successful.')

In [ ]:
# === Cell 5: 数据获取 ===
stock = get_stock_daily(STOCK_MOUTAI, DEFAULT_START, DEFAULT_END)
benchmark = get_index_daily(CSI300_CODE, DEFAULT_START, DEFAULT_END)

close = stock['close']
high = stock['high']
low = stock['low']
volume = stock['volume']

print(f'Stock data: {len(stock)} trading days')

In [ ]:
# === Cell 6: 特征工程 + 专家策略 ===

# State features
feat = pd.DataFrame(index=stock.index)
feat['ret_1d'] = close.pct_change(1)
feat['ret_5d'] = close.pct_change(5)
feat['vol_10d'] = feat['ret_1d'].rolling(10).std()
feat['rsi_14'] = rsi(close, 14)
feat['ma_dev_20'] = (close - close.rolling(20).mean()) / close.rolling(20).mean()
feat['range'] = (high - low) / close
macd_df = macd(close)
feat['macd_hist'] = macd_df['histogram']

feat_cols = ['ret_1d', 'ret_5d', 'vol_10d', 'rsi_14', 'ma_dev_20', 'range', 'macd_hist']
feat.dropna(inplace=True)

# Normalize
for col in feat_cols:
    feat[col] = (feat[col] - feat[col].rolling(60).mean()) / (feat[col].rolling(60).std() + 1e-8)
feat.dropna(inplace=True)

# === Expert Strategy: MA(10,30) crossover (profitable baseline) ===
ma_fast = sma(close, 10)
ma_slow = sma(close, 30)
expert_signals = pd.Series(0, index=close.index)
expert_signals[ma_fast > ma_slow] = 1
expert_signals = expert_signals.reindex(feat.index).fillna(0).astype(int)

print(f'Features: {feat_cols}')
print(f'Expert strategy: MA(10,30) crossover')
print(f'Expert long ratio: {expert_signals.mean():.2%}')

In [ ]:
# === Cell 7: MaxEnt IRL ===

feature_matrix = feat[feat_cols].values
n_days = len(feature_matrix)
n_features = len(feat_cols)

# Expert feature expectations (average features under expert actions)
expert_actions = expert_signals.values
expert_feat_expectations = np.zeros(n_features)
for t in range(n_days):
    if expert_actions[t] == 1:
        expert_feat_expectations += feature_matrix[t]
expert_feat_expectations /= max(expert_actions.sum(), 1)

print('Expert feature expectations:')
for i, col in enumerate(feat_cols):
    print(f'  {col}: {expert_feat_expectations[i]:.4f}')

# MaxEnt IRL: iterate to match feature expectations
theta = np.zeros(n_features)  # Reward weights
lr_irl = 0.01
n_irl_iters = 100

theta_history = [theta.copy()]
feat_match_errors = []

for iteration in range(n_irl_iters):
    # Compute reward for each state
    rewards = feature_matrix @ theta  # R(s) = theta^T * phi(s)

    # Soft policy (MaxEnt): P(a=1|s) proportional to exp(R(s))
    exp_r = np.exp(np.clip(rewards, -10, 10))
    policy_probs = exp_r / (1 + exp_r)  # Sigmoid

    # Learner feature expectations under current soft policy
    learner_feat_expectations = np.zeros(n_features)
    for t in range(n_days):
        learner_feat_expectations += policy_probs[t] * feature_matrix[t]
    learner_feat_expectations /= max(policy_probs.sum(), 1)

    # Gradient: expert - learner
    gradient = expert_feat_expectations - learner_feat_expectations
    theta += lr_irl * gradient
    theta_history.append(theta.copy())

    error = np.linalg.norm(gradient)
    feat_match_errors.append(error)

    if (iteration + 1) % 20 == 0:
        print(f'IRL iter {iteration+1}: feature match error = {error:.6f}')

print(f'\nLearned reward weights:')
for i, col in enumerate(feat_cols):
    print(f'  {col}: {theta[i]:.4f}')

In [ ]:
# === Cell 8: 用IRL学到的奖励函数训练RL智能体 ===

class SimpleQNet:
    def __init__(self, state_dim, n_actions=2, lr=0.001):
        np.random.seed(42)
        self.W1 = np.random.randn(state_dim, 32) * 0.1
        self.b1 = np.zeros(32)
        self.W2 = np.random.randn(32, n_actions) * 0.1
        self.b2 = np.zeros(n_actions)
        self.lr = lr

    def predict(self, s):
        if s.ndim == 1: s = s.reshape(1, -1)
        h = np.maximum(0, s @ self.W1 + self.b1)
        return h @ self.W2 + self.b2

    def update(self, states, actions, targets):
        h = np.maximum(0, states @ self.W1 + self.b1)
        q = h @ self.W2 + self.b2
        dq = np.zeros_like(q)
        for i, a in enumerate(actions):
            dq[i, a] = (q[i, a] - targets[i]) / len(actions)
        dW2 = h.T @ dq
        db2 = dq.sum(0)
        dh = dq @ self.W2.T
        dh[h <= 0] = 0
        self.W2 -= self.lr * dW2
        self.b2 -= self.lr * db2
        self.W1 -= self.lr * (states.T @ dh)
        self.b1 -= self.lr * dh.sum(0)

    def copy_from(self, other):
        self.W1, self.b1 = other.W1.copy(), other.b1.copy()
        self.W2, self.b2 = other.W2.copy(), other.b2.copy()


# Train agent with IRL-learned reward
train_end = int(n_days * 0.7)
learned_rewards = feature_matrix @ theta  # IRL reward signal

state_dim = n_features + 1  # features + position
q_net = SimpleQNet(state_dim, 2, lr=0.001)
target_net = SimpleQNet(state_dim, 2, lr=0.001)
target_net.copy_from(q_net)

buffer = deque(maxlen=3000)
epsilon = 1.0
gamma = 0.95
n_episodes = 12
ep_rewards_irl = []

for ep in range(n_episodes):
    pos = 0
    total_r = 0
    for t in range(train_end - 1):
        state = np.append(feature_matrix[t], pos)
        if np.random.random() < epsilon:
            action = np.random.randint(2)
        else:
            action = np.argmax(q_net.predict(state)[0])

        # Use IRL-learned reward instead of raw return
        reward = learned_rewards[t + 1] * (1 if action == 1 else -0.1)
        next_state = np.append(feature_matrix[t + 1], action)
        done = (t == train_end - 2)

        buffer.append((state, action, reward, next_state, done))

        if len(buffer) >= 64:
            batch = random.sample(buffer, 64)
            s_b, a_b, r_b, ns_b, d_b = zip(*batch)
            s_b, a_b, r_b, ns_b, d_b = (np.array(s_b), np.array(a_b),
                                          np.array(r_b), np.array(ns_b), np.array(d_b))
            next_q = target_net.predict(ns_b)
            targets = r_b + gamma * np.max(next_q, 1) * (1 - d_b)
            q_net.update(s_b, a_b.astype(int), targets)

        if t % 50 == 0:
            target_net.copy_from(q_net)

        pos = action
        total_r += reward

    epsilon = max(0.05, epsilon * 0.9)
    ep_rewards_irl.append(total_r)
    print(f'Episode {ep+1}: IRL reward = {total_r:.4f}, eps = {epsilon:.3f}')

In [ ]:
# === Cell 9: 回测对比 (专家 vs IRL智能体) ===

# IRL agent signals
irl_actions = []
pos = 0
for t in range(n_days):
    state = np.append(feature_matrix[t], pos)
    action = np.argmax(q_net.predict(state)[0])
    irl_actions.append(action)
    pos = action

irl_signals = pd.Series(irl_actions, index=feat.index, name='irl_signal')

# Test period only
test_prices = close.reindex(feat.index).iloc[train_end:]
bench_close = benchmark['close'] if 'close' in benchmark.columns else None
bt = Backtester(initial_capital=INITIAL_CAPITAL, t_plus_1=True)

# Expert backtest
result_expert = bt.run(test_prices, expert_signals.iloc[train_end:], bench_close)
# IRL agent backtest
result_irl = bt.run(test_prices, irl_signals.iloc[train_end:], bench_close)

print('=== Expert (MA 10/30) ===')
for k, v in result_expert['metrics'].items():
    print(f'  {k}: {v}')

print('\n=== IRL Agent ===')
for k, v in result_irl['metrics'].items():
    print(f'  {k}: {v}')

In [ ]:
# === Cell 10: 可视化 ===

# 1. IRL convergence
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(feat_match_errors, 'b-', linewidth=1.5)
axes[0].set_title('IRL 特征匹配误差收敛', fontsize=13, fontweight='bold')
axes[0].set_xlabel('迭代次数')
axes[0].set_ylabel('||grad||')
axes[0].grid(True, alpha=0.3)

# Reward weights evolution
theta_arr = np.array(theta_history)
for i, col in enumerate(feat_cols):
    axes[1].plot(theta_arr[:, i], label=col, linewidth=1.2)
axes[1].set_title('奖励权重 theta 演化', fontsize=13, fontweight='bold')
axes[1].set_xlabel('迭代次数')
axes[1].legend(fontsize=8, ncol=2)
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 2. Learned reward weights bar chart
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#4CAF50' if w > 0 else '#F44336' for w in theta]
ax.barh(feat_cols, theta, color=colors, alpha=0.8)
ax.set_title('IRL 学到的奖励权重', fontsize=14, fontweight='bold')
ax.set_xlabel('权重值')
ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

# 3. Cumulative comparison
plot_cumulative_comparison(
    {'Expert (MA10/30)': result_expert['returns'], 'IRL Agent': result_irl['returns']},
    title='专家策略 vs IRL模仿智能体 - 累计收益'
)

# 4. RL trading animation
test_actions_irl = irl_actions[train_end:]
returns_test = feat['ret_1d'].reindex(feat.index).fillna(0).values[train_end:]
fig_anim = animate_rl_trading(
    test_prices.values, test_actions_irl, returns_test,
    title='IRL 模仿智能体交易过程'
)
fig_anim.show()

# 5. Metrics
plot_equity_curve(result_irl['equity_curve'], title='IRL Agent - 权益曲线')
plot_drawdown(result_irl['equity_curve'], title='IRL Agent - 回撤')
plot_metrics_table(result_irl['metrics'], title='IRL Agent - 绩效指标')

## 结果讨论

### 核心发现
1. **奖励函数学习**: MaxEnt IRL成功从专家交易轨迹中推断出有意义的奖励权重
2. **特征重要性**: IRL学到的权重揭示了专家策略隐含的决策逻辑（如偏好正动量、低波动环境）
3. **模仿效果**: IRL智能体能在测试期复现专家策略的交易风格，且可能通过泛化获得改进

### 与论文对比
- Zhang (2023) 针对T+1约束设计了专门的状态表示和动作空间
- 本实现使用简化的MaxEnt IRL + DQN框架
- 核心创新：从"模仿what"到"理解why" -- 学习奖励函数而非直接模仿动作

### IRL vs 直接模仿学习
- **行为克隆 (BC)**: 直接学习 $\pi(a|s) \approx \pi_E(a|s)$ → 分布偏移问题
- **IRL**: 学习 $R(s,a)$ → 更鲁棒，可适应环境变化

### 改进方向
- 使用GAIL (Generative Adversarial Imitation Learning) 替代MaxEnt IRL
- 多专家融合: 从多个盈利策略中学习共性奖励函数
- 考虑T+1约束对奖励函数设计的影响